###Dashboard Visualizations
####Generates the Gold layer dashboard - produces 12 matplotlib charts across Orders, Reviews, and Customers sections; exports a multi-page PDF report to ADLS Gen2 reports container

In [0]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.backends.backend_pdf as pdf_backend
import seaborn as sns
import pandas as pd
import numpy as np

In [0]:
#GLOBAL THEME & COLOR PALETTE
#Apply dark theme across all charts for consistent PDF styling
#COLORS: general purpose palette for bars, lines, pies
#SENTIMENT_COLORS: fixed color mapping so positive/neutral/negative are always green/orange/red across all sentiment charts
plt.rcParams.update({
    'figure.facecolor': '#1e1e2e',
    'axes.facecolor': '#1e1e2e',
    'axes.edgecolor': '#444',
    'axes.labelcolor': 'white',
    'xtick.color': 'white',
    'ytick.color': 'white',
    'text.color': 'white',
    'grid.color': '#333',
    'grid.linestyle': '--',
    'grid.alpha': 0.5,
    'font.size': 11
})

COLORS = ['#4e79a7', '#f28e2b', '#59a14f', '#e15759', '#76b7b2', '#edc948']
SENTIMENT_COLORS = {
    'positive': '#59a14f',
    'neutral':  '#f28e2b',
    'negative': '#e15759'
}

In [0]:
%run /Workspace/e-commerce/Parameter_setup

In [0]:
#LOAD GOLD LAYER TABLES 
#Read all three Gold tables once into Pandas DataFrames
#orders_df     : fact table — joined with current customers (is_current = TRUE)
#reviews_df    : reviews with Azure AI sentiment scores
#customers_df  : current active customers only (is_current = TRUE)
#customers_scd_df : all versions including historical — used for SCD2 charts
#Date columns converted to datetime and period strings for time-series grouping

orders_df = spark.sql("""
    SELECT
        o.order_item_id, o.order_id, o.order_status,
        o.category_name_english, o.seller_city, o.seller_state,
        o.price, o.freight_value, o.total_revenue,
        o.payment_value, o.payment_installments, o.payment_type,
        o.delivery_days, o.estimated_delivery_days, o.delivery_delay_days,
        o.order_purchase_date,
        c.state AS customer_state, c.city AS customer_city
    FROM ecom_prep.orders o
    JOIN ecom_prep.customers c
      ON o.customer_key = c.customer_key
     AND c.is_current = TRUE
""").toPandas()

reviews_df = spark.sql("""
    SELECT
        review_id, order_id,
        review_score, review_comment_message,
        sentiment, positive_score, neutral_score, negative_score,
        review_creation_date
    FROM ecom_prep.reviews
""").toPandas()

customers_df = spark.sql("""
    SELECT * FROM ecom_prep.customers WHERE is_current = TRUE
""").toPandas()

customers_scd_df = spark.sql("""
    SELECT * FROM ecom_prep.customers
""").toPandas()

# Date conversions
orders_df['order_purchase_date'] = pd.to_datetime(orders_df['order_purchase_date'])
orders_df['order_month'] = orders_df['order_purchase_date'].dt.to_period('M').astype(str)

reviews_df['review_creation_date'] = pd.to_datetime(reviews_df['review_creation_date'])
reviews_df['review_month'] = reviews_df['review_creation_date'].dt.to_period('M').astype(str)

customers_scd_df['start_date'] = pd.to_datetime(customers_scd_df['start_date'])
customers_scd_df['start_month'] = customers_scd_df['start_date'].dt.to_period('M').astype(str)

print("All tables loaded successfully")
print(f"   Orders    : {len(orders_df):,} rows")
print(f"   Reviews   : {len(reviews_df):,} rows")
print(f"   Customers : {len(customers_df):,} rows (current)")
print(f"   SCD Total : {len(customers_scd_df):,} rows (all versions)")


In [0]:
#OPEN PDF 
pdf = pdf_backend.PdfPages(LOCAL_PDF_PATH)

#SECTION HEADER HELPER
def section_page(title, subtitle=''):
    fig = plt.figure(figsize=(14, 3))
    fig.patch.set_facecolor('#12121f')
    fig.text(0.5, 0.62, title, fontsize=28, fontweight='bold', ha='center', color='white')
    if subtitle:
        fig.text(0.5, 0.25, subtitle, fontsize=14, ha='center', color='#aaa')
    pdf.savefig(fig, bbox_inches='tight')
    plt.close(fig)

#COVER PAGE 
fig = plt.figure(figsize=(14, 8))
fig.patch.set_facecolor('#1e1e2e')
fig.text(0.5, 0.65, 'E-Commerce', fontsize=36, fontweight='bold', ha='center', color='white')
fig.text(0.5, 0.55, 'Data Engineering Portfolio Project', fontsize=20, ha='center', color='#aaa')
fig.text(0.5, 0.43, 'Azure Data Factory  •  Databricks  •  Delta Lake', fontsize=14,
         ha='center', color='#4e79a7')
fig.text(0.5, 0.36, 'Unity Catalog  •  Azure AI Language  •  Medallion Architecture', fontsize=14,
         ha='center', color='#4e79a7')
fig.text(0.5, 0.23, 'Gold Layer Dashboard - 2018', fontsize=16, ha='center', color='#f28e2b', fontweight='bold')
fig.text(0.5, 0.11, 'Neha Pattankar  |  linkedin.com/in/nehapattankar/', fontsize=11,
         ha='center', color='#888')
pdf.savefig(fig, bbox_inches='tight')
plt.close(fig)

###Analysing Orders

In [0]:
section_page('Orders Analysis', '5 Visualizations — Trends, Revenue, Categories & Delivery')

#1A. Monthly Order Volume Trend
#Line chart with area fill showing order count per month
#Reveals seasonality and overall business growth trend
#x-axis: order_month (derived from order_purchase_date in Gold layer)

monthly_orders = orders_df.groupby('order_month')['order_id'].nunique().reset_index()
monthly_orders.columns = ['month', 'order_count']

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly_orders['month'], monthly_orders['order_count'],
        color='#4e79a7', linewidth=2.5, marker='o', markersize=5)
ax.fill_between(monthly_orders['month'], monthly_orders['order_count'],
                alpha=0.15, color='#4e79a7')
ax.set_title('1A — Monthly Order Volume Trend', fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Month')
ax.set_ylabel('Number of Orders')
ax.tick_params(axis='x', rotation=45)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.grid(True)
plt.tight_layout()
pdf.savefig(fig, bbox_inches='tight')
plt.show()
plt.close(fig)

In [0]:
#1B. Order Status Distribution 
#Side-by-side: horizontal bar chart (absolute counts) + pie chart (percentage)
#Deduplicated on order_id to avoid double-counting multi-item orders
#Shows what % of orders are delivered vs cancelled/unavailable

status_counts = orders_df.drop_duplicates('order_id')['order_status'].value_counts().reset_index()
status_counts.columns = ['status', 'count']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].barh(status_counts['status'], status_counts['count'],
             color=COLORS[:len(status_counts)])
axes[0].set_title('Order Status — Count', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Orders')
for i, v in enumerate(status_counts['count']):
    axes[0].text(v + 50, i, f'{v:,}', va='center', fontsize=9)

axes[1].pie(status_counts['count'], labels=status_counts['status'],
            colors=COLORS[:len(status_counts)], autopct='%1.1f%%',
            startangle=90, pctdistance=0.82)
axes[1].set_title('Order Status — Distribution %', fontsize=13, fontweight='bold')
plt.suptitle('1B — Order Status Overview', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
pdf.savefig(fig, bbox_inches='tight')
plt.show()
plt.close(fig)

In [0]:
#1C. Revenue by Customer State (Top 10)
#Bar chart of total payment_value grouped by customer state
#Highlights which Brazilian states drive the most revenue
#Values formatted in BRL (Brazilian Real)

revenue_by_state = orders_df.groupby('customer_state')['payment_value'].sum().reset_index()
revenue_by_state = revenue_by_state.sort_values('payment_value', ascending=False).head(10)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(revenue_by_state['customer_state'], revenue_by_state['payment_value'],
              color=COLORS[0], edgecolor='white', linewidth=0.5)
ax.set_title('1C — Top 10 Customer States by Revenue', fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('State')
ax.set_ylabel('Total Revenue (BRL)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x/1e6:.1f}M'))
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000,
            f'R${bar.get_height()/1e6:.1f}M', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
pdf.savefig(fig, bbox_inches='tight')
plt.show()
plt.close(fig)

In [0]:
#1D. Top 10 Product Categories by Orders
#Horizontal bar chart — sorted ascending so largest bar is at top
#Uses category_name_english (English translation from reference table)
#Distinct order_id count per category to avoid item-level inflation

category_orders = orders_df.groupby('category_name_english')['order_id'].nunique().reset_index()
category_orders = category_orders.sort_values('order_id', ascending=True).tail(10)

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(category_orders['category_name_english'],
        category_orders['order_id'], color=COLORS[1])
ax.set_title('1D — Top 10 Product Categories by Orders', fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Number of Orders')
for bar in ax.patches:
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2,
            f'{int(bar.get_width()):,}', va='center', fontsize=9)
plt.tight_layout()
pdf.savefig(fig, bbox_inches='tight')
plt.show()
plt.close(fig)

In [0]:
#1E. Delivery Performance Analysis 
#delivery_days and delivery_delay_days are pre-computed in Gold orders table
#Filters to 'delivered' orders with non-null delivery_days only
#Left chart : avg delivery days by state (top 10 slowest)
#Right chart: avg delay days by state — positive = late, negative = early

delivered = orders_df[
    (orders_df['order_status'] == 'delivered') &
    (orders_df['delivery_days'].notna())
].copy()

avg_delivery = delivered.groupby('customer_state')['delivery_days'].mean().reset_index()
avg_delay    = delivered.groupby('customer_state')['delivery_delay_days'].mean().reset_index()
avg_delivery = avg_delivery.sort_values('delivery_days', ascending=False).head(10)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(avg_delivery['customer_state'], avg_delivery['delivery_days'],
            color=COLORS[3], edgecolor='white', linewidth=0.5)
axes[0].set_title('Avg Delivery Days — Top 10 Slowest States', fontsize=12, fontweight='bold')
axes[0].set_xlabel('State')
axes[0].set_ylabel('Avg Delivery Days')
for bar in axes[0].patches:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 f'{bar.get_height():.1f}d', ha='center', va='bottom', fontsize=8)

delay_summary = pd.DataFrame({
    'category': ['On Time', 'Delayed'],
    'count': [
        (delivered['delivery_delay_days'] <= 0).sum(),
        (delivered['delivery_delay_days'] > 0).sum()
    ]
})
axes[1].pie(delay_summary['count'], labels=delay_summary['category'],
            colors=[COLORS[2], COLORS[3]], autopct='%1.1f%%',
            startangle=90, pctdistance=0.82)
axes[1].set_title('On Time vs Delayed Deliveries', fontsize=12, fontweight='bold')

plt.suptitle('1E — Delivery Performance', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
pdf.savefig(fig, bbox_inches='tight')
plt.show()
plt.close(fig)

###Analyzing Reviews

In [0]:
section_page('Reviews & Sentiment Analysis', 'Powered by Azure AI Language Service — 4 Visualizations')

#2A. Sentiment Distribution (HERO CHART) 
#Side-by-side: pie chart (proportion) + bar chart (absolute counts)
#Uses fixed SENTIMENT_COLORS so positive=green, neutral=orange, negative=red
#Hero chart — most impactful visual showing Azure AI integration result

sentiment_counts = reviews_df['sentiment'].str.lower().value_counts().reset_index()
sentiment_counts.columns = ['sentiment', 'count']
colors = [SENTIMENT_COLORS.get(s, '#888') for s in sentiment_counts['sentiment']]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
wedges, texts, autotexts = axes[0].pie(
    sentiment_counts['count'],
    labels=sentiment_counts['sentiment'].str.capitalize(),
    colors=colors, autopct='%1.1f%%', startangle=90,
    pctdistance=0.75, wedgeprops=dict(width=0.5))
for at in autotexts:
    at.set_fontsize(11)
    at.set_fontweight('bold')
axes[0].set_title('Sentiment Distribution\n(Azure AI Language)', fontsize=13, fontweight='bold')

bars = axes[1].bar(sentiment_counts['sentiment'].str.capitalize(),
                   sentiment_counts['count'], color=colors, edgecolor='white', width=0.5)
axes[1].set_title('Sentiment Count', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Number of Reviews')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for bar in bars:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=11, fontweight='bold')
plt.suptitle('2A — ⭐ Customer Sentiment Analysis — Powered by Azure AI Language',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
pdf.savefig(fig, bbox_inches='tight')
plt.show()
plt.close(fig)

In [0]:
#2B. Sentiment Trend Over Time 
#Multi-line chart — one line per sentiment label across review_month
#Pivot on sentiment so each label becomes its own column
#Shows whether customer satisfaction improved or declined over time

sentiment_monthly = reviews_df.groupby(
    ['review_month', reviews_df['sentiment'].str.lower()]
).size().reset_index(name='count')
sentiment_monthly.columns = ['review_month', 'sentiment', 'count']
sentiment_pivot = sentiment_monthly.pivot(
    index='review_month', columns='sentiment', values='count'
).fillna(0)

fig, ax = plt.subplots(figsize=(14, 5))
for col in sentiment_pivot.columns:
    color = SENTIMENT_COLORS.get(col.lower(), '#888')
    ax.plot(sentiment_pivot.index, sentiment_pivot[col],
            label=col.capitalize(), color=color, linewidth=2.5, marker='o', markersize=4)
ax.set_title('2B — Sentiment Trend Over Time', fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Month')
ax.set_ylabel('Number of Reviews')
ax.legend(loc='upper left')
ax.tick_params(axis='x', rotation=45)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.grid(True)
plt.tight_layout()
pdf.savefig(fig, bbox_inches='tight')
plt.show()
plt.close(fig)

In [0]:
#2C. Review Score vs Sentiment Heatmap 
#Seaborn heatmap - rows: review_score (1-5), columns: sentiment label
#Validates Azure AI output: score 5 should be mostly positive,
#score 1 should be mostly negative — confirms API accuracy

score_sentiment = reviews_df.groupby(
    ['review_score', reviews_df['sentiment'].str.lower()]
).size().reset_index(name='count')
score_sentiment.columns = ['review_score', 'sentiment', 'count']
score_pivot = score_sentiment.pivot(
    index='review_score', columns='sentiment', values='count'
).fillna(0)

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(score_pivot, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Count'})
ax.set_title('2C — Review Score vs Sentiment', fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Sentiment')
ax.set_ylabel('Review Score (1-5)')
plt.tight_layout()
pdf.savefig(fig, bbox_inches='tight')
plt.show()
plt.close(fig)

In [0]:
#2D. Azure AI Confidence Scores 
#Grouped bar chart — avg positive/neutral/negative confidence score
#broken down by the actual sentiment label assigned by the API
#Each trio of bars should show high confidence for the correct label
#e.g. rows labelled 'positive' should have high positive_score

reviews_df['sentiment_lower'] = reviews_df['sentiment'].str.lower()
conf_df = reviews_df.groupby('sentiment_lower')[
    ['positive_score', 'neutral_score', 'negative_score']
].mean().reset_index()

x = np.arange(len(conf_df))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width, conf_df['positive_score'], width, label='Positive Score', color=SENTIMENT_COLORS['positive'])
ax.bar(x,         conf_df['neutral_score'],  width, label='Neutral Score',  color=SENTIMENT_COLORS['neutral'])
ax.bar(x + width, conf_df['negative_score'], width, label='Negative Score', color=SENTIMENT_COLORS['negative'])

ax.set_title('2D — Azure AI Confidence Scores by Sentiment Label', fontsize=13, fontweight='bold', pad=15)
ax.set_ylabel('Avg Confidence Score (0–1)')
ax.set_xticks(x)
ax.set_xticklabels(conf_df['sentiment_lower'].str.capitalize())
ax.set_ylim(0, 1.15)
ax.legend()
ax.grid(True, axis='y')
for bars_group in ax.containers:
    ax.bar_label(bars_group, fmt='%.2f', padding=3, fontsize=9)
plt.tight_layout()
pdf.savefig(fig, bbox_inches='tight')
plt.show()
plt.close(fig)

###Customers SCD

In [0]:
section_page('Customer Analysis', 'SCD Type 2 Implementation — 3 Visualizations')

#3A. Active Customers by State (Top 10) 
#Bar chart using customers_df (is_current = TRUE only)
#Shows geographic distribution of active customer base

customers_by_state = customers_df['state'].value_counts().head(10).reset_index()
customers_by_state.columns = ['state', 'count']

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(customers_by_state['state'], customers_by_state['count'],
              color=COLORS[0], edgecolor='white', linewidth=0.5)
ax.set_title('3A — Top 10 States by Active Customers', fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('State')
ax.set_ylabel('Number of Customers')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
pdf.savefig(fig, bbox_inches='tight')
plt.show()
plt.close(fig)

In [0]:
#3B. Customer Growth Over Time (SCD2 insight) 
#Dual-axis chart: bars = new unique customers per month, line = cumulative total
#Uses customers_scd_df (all versions) and start_date to track when each
#customer_unique_id first appeared — pure SCD2 derived insight

customer_growth = customers_scd_df.groupby(
    'start_month'
)['customer_unique_id'].nunique().reset_index()
customer_growth.columns = ['month', 'new_customers']
customer_growth['cumulative'] = customer_growth['new_customers'].cumsum()

fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()
ax1.bar(customer_growth['month'], customer_growth['new_customers'],
        color=COLORS[0], alpha=0.6, label='New Customers/Month')
ax2.plot(customer_growth['month'], customer_growth['cumulative'],
         color=COLORS[2], linewidth=2.5, marker='o', markersize=4, label='Cumulative')
ax1.set_title('3B — Customer Growth Over Time (SCD Type 2)', fontsize=15, fontweight='bold', pad=15)
ax1.set_xlabel('Month')
ax1.set_ylabel('New Customers', color=COLORS[0])
ax2.set_ylabel('Cumulative Customers', color=COLORS[2])
ax1.tick_params(axis='x', rotation=45)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.tight_layout()
pdf.savefig(fig, bbox_inches='tight')
plt.show()
plt.close(fig)

In [0]:
#3C. SCD Type 2: Current vs Historical Records 
#Bar chart comparing is_current = TRUE vs FALSE record counts
#Directly visualises SCD2 implementation — historical rows are customers
#who changed city/state and got a new version record in Gold layer
#pdf.close() called here — finalises and saves the PDF to LOCAL_PDF_PATH
scd_stats = customers_scd_df.groupby('is_current').size().reset_index(name='count')
scd_stats['label'] = scd_stats['is_current'].map({True: 'Current Records', False: 'Historical Records'})

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(scd_stats['label'], scd_stats['count'],
              color=[COLORS[2], COLORS[3]], edgecolor='white', width=0.4)
ax.set_title('3C — SCD Type 2: Current vs Historical Records', fontsize=13, fontweight='bold', pad=15)
ax.set_ylabel('Number of Records')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=12, fontweight='bold')
plt.tight_layout()
pdf.savefig(fig, bbox_inches='tight')
plt.show()
plt.close(fig)

In [0]:
# CLOSE PDF
pdf.close()
print(f"PDF saved locally: {LOCAL_PDF_PATH}")

In [0]:
#UPLOAD PDF TO ADLS 
#Copy completed PDF from Workspace local path to ADLS reports container
#using dbutils.fs.cp with file: URI prefix
#file: prefix required to reference Workspace local filesystem in dbutils

dbutils.fs.cp(
    f"file:{LOCAL_PDF_PATH}",
    ADLS_PDF_PATH,
    True
)
print(f"PDF uploaded to ADLS: {ADLS_PDF_PATH}")